# Segment 1 — LCEL basics

In LangChain you glue pieces together with the `|` pipe.
Data flows left to right: **prompt → llm → parser**.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## Build a chain
A prompt with `{placeholders}`, the model, and a parser that returns plain text.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Answer in one sentence."),
    ("human", "Explain {topic} to a {audience}."),
])
parser = StrOutputParser()

chain = prompt | llm | parser

## Run it once with `invoke`

In [ ]:
chain.invoke({"topic": "vector embeddings", "audience": "five year old"})

## Same chain, streamed word by word

In [ ]:
for chunk in chain.stream({"topic": "LCEL", "audience": "backend engineer"}):
    print(chunk, end="")